In [1]:
import torch
from torch_geometric.data import HeteroData
from models.hetero_gnn import TripartiteHeteroGNN



ModuleNotFoundError: No module named 'torch'

In [ ]:
# 1. Define a Dummy Linear Problem (LP)
# ==========================================
# Goal: Minimize c^T x  subject to  Ax <= b
# 3 Variables, 2 Constraints
n_vars = 3
n_cons = 2

# Constraint Matrix A (2x3)
A = torch.tensor([
    [1.0, 2.0, 0.0],  # Constraint 1: 1*x0 + 2*x1 + 0*x2
    [0.0, 1.0, 1.0]   # Constraint 2: 0*x0 + 1*x1 + 1*x2
])

# Bounds b (size 2)
b = torch.tensor([10.0, 5.0])

# Cost vector c (size 3)
c = torch.tensor([1.5, 1.0, 2.0])


In [ ]:

# ==========================================
# 2. Build the HeteroData Object
# ==========================================
# The TripartiteHeteroGNN expects a graph with 'vals', 'cons', and 'obj' nodes.
data = HeteroData()

# --- A. Node Features ---
# The model expects input features (e.g., mean/std of rows/cols). 
# We'll use random features of size 2 for this example.
input_feature_dim = 2
data['vals'].x = torch.randn(n_vars, input_feature_dim)
data['cons'].x = torch.randn(n_cons, input_feature_dim)
data['obj'].x = torch.randn(1, input_feature_dim)  # Only 1 objective node

# --- B. Edge Connectivity & Attributes ---
# We must define edges for all bipartite connections: 
# (vals<->cons), (vals<->obj), (cons<->obj)

# 1. Variables to Constraints (based on A matrix)
rows, cols = A.nonzero(as_tuple=True)
# Edge: vals -> cons
data['vals', 'to', 'cons'].edge_index = torch.stack([cols, rows], dim=0)
data['vals', 'to', 'cons'].edge_attr = A[rows, cols].view(-1, 1) # Weight A_ij

# Edge: cons -> vals (Inverse)
data['cons', 'to', 'vals'].edge_index = torch.stack([rows, cols], dim=0)
data['cons', 'to', 'vals'].edge_attr = A[rows, cols].view(-1, 1)

# 2. Variables to Objective (Fully connected)
v_idx = torch.arange(n_vars)
o_idx = torch.zeros(n_vars, dtype=torch.long)
# Edge: vals -> obj (Weight: c_j)
data['vals', 'to', 'obj'].edge_index = torch.stack([v_idx, o_idx], dim=0)
data['vals', 'to', 'obj'].edge_attr = c.view(-1, 1)

# Edge: obj -> vals
data['obj', 'to', 'vals'].edge_index = torch.stack([o_idx, v_idx], dim=0)
data['obj', 'to', 'vals'].edge_attr = c.view(-1, 1)

# 3. Constraints to Objective (Fully connected)
c_idx = torch.arange(n_cons)
o_idx_c = torch.zeros(n_cons, dtype=torch.long)
# Edge: cons -> obj (Weight: b_i)
data['cons', 'to', 'obj'].edge_index = torch.stack([c_idx, o_idx_c], dim=0)
data['cons', 'to', 'obj'].edge_attr = b.view(-1, 1)

# Edge: obj -> cons
data['obj', 'to', 'cons'].edge_index = torch.stack([o_idx_c, c_idx], dim=0)
data['obj', 'to', 'cons'].edge_attr = b.view(-1, 1)



In [ ]:
# ==========================================
# 3. Initialize the Model with GCNConv
# ==========================================
# We use the class from models/hetero_gnn.py
model = TripartiteHeteroGNN(
    conv='gcnconv',           # <--- Selects GCNConv from models/gcnconv.py
    in_shape=input_feature_dim,
    pe_dim=0,                 # No Positional Encoding for simplicity
    hid_dim=16,               # Hidden dimension
    num_conv_layers=2,        # Number of Message Passing Layers
    num_pred_layers=2,        # MLP layers for final prediction
    num_mlp_layers=1,         # MLP layers inside the GCNConv
    dropout=0.0,
    share_conv_weight=False,
    share_lin_weight=False,
    use_norm=True,
    use_res=False,
    conv_sequence='parallel'  # Can also use 'cvo' for sequential updates
)

# ==========================================
# 4. Forward Pass
# ==========================================
# The model outputs predictions for variables and constraints
vals_pred, cons_pred = model(data)

print("Predicted Variable Values (z):")
print(vals_pred)
# Shape: [1, n_vars] -> e.g., predictions for x0, x1, x2